   ... normal to ... Computer architecture is notoriously dense, with a lot of
   moving parts between the software we write and the physical silicon that 
   executes it.

   ... high-level refresher of the first two lectures to help you get your 
   bearings again.


---   
LECTURE 1: ARCHITECTURE AND INSTRUCTIONS
   This lecture sets the stage for the entire module. It answers the fundamental
   question of why we care about hardware when we usually just write code, and
   introduces the bridge between the two.

   - THE BIG PICTURE: The core concept here is the INSTRUCTION SET ARCHITECTURE
     (ISA). The ISA is the critical interface or "contract" between software
     and hardware. It defines the set of instructions the processor can
     understand. 
   - WHY ARCHITECTURE MATTERS: The lecture motivates this by looking at modern
     accelerators. Standard CPU's aren't always enough anymore. The timestamps
     mention TPUs (Tensor Processing Units), which are highly specialised chips
     designed explicitly to accelerate matrix multiplications for machine
     learning models (like the Transformers powering GPT). Cloud providers
     (Microsoft, Amazon) rely heavily on these custom architectures.
   - THE RISC-V ARCHITECTURE: This module focuses on RISC-V (Reduced Instruction
     Set Computer). It is an open-source ISA that prioritises simplicity and
     efficiency.
      - R-TYPE INSTRUCTIONS: These are "Register-type" instructions used for
        standard arithmetic and logical operations (like `add`, `sub`, `and`).
        They typically take two source registers, perform an operation, and
        store the result in a destination register. 
      - FROM CODE TO SILICON: You looked at how high-level control flow (like an
        `if`-statement in C or C++) gets compiled down itno a sequence of these 
        basic RISC-V instructions (using branches and jumps).
      - THE DATAPATH: This is the physical wiring, ALUs, and memory components
        that actually execute the instructions. 





---
ISA: Instruction Set Architecture
   The ISA is the INTERFACE between software and hardware -- like an API for the
   processor. It defines:

   * What INSTRUCTIONS the processor understands (add, subtract, load, store,
     jump...)
   * What REGISTERS are available (small, fast storage inside the processor)
   * How MEMORY is addressed
   * The ENCODING FORMAT of instructions (how they're represented in binary)

   The key insight: DIFFERENT HARDWARE CAN IMPLEMENT THE SAME ISA. Your laptop
   and a server can both run the same RISC-V code, even if their internal
   circuits are completely different. This is what makes software portable.


DESIGN APPROACHES: RISC vs. CISC
   There are two major philosophies:


CISC (Complex Instruction Set Computer) -- e.g., x86 (Intel/AMD). Instructions
   can be very complex ("Multiply these two memory values and store the result
   back to memory in one instruction"). Instructions vary in length and can take
   many cycles.


RISC (Reduces Instruction Set Computer) -- e.g., RISC-V, ARM. Instructions are
   simple, fixed-length, and typically execute in one cycle. If you need 
   something complex, you combine multiple simple instructions. This makes the
   hardware much simpler and easier to pipeline (which you'll cover later).


   Your module uses RISC-V as the teaching architecture because it's clean, 
   open-source, and designed specifically to be easy to understand. 



---
RISC-V Architecture Basics
   RISC-V has 32 REGISTERS (x0 through x31), each 32 or 64 bits wide. Register
   x0 is hardwired to ALWAYS BE ZERO -- this is surprisingly useful (you'll
   see why). Instructions operate on registers, not directly on memory, 
   following the LOAD-STORE principle: you load data from memory into registers,
   do your computation, then store results back.


RISC-V R-type Instructions
   R-type instructions operate on THREE REGISTERS: two sources and one 
   destination. The format is:
```
add x5, x6, x7          # x5 = x6 + x7
sub x5, x6, x7          # x5 = x6 - x7
```

   The binary encoding has fixed fields:
```
|  funct7 (7)  |  rs2 (5)  |  rs1  (5)  |  funct3 (3)  |  rd (5)  | opcode (7)  |
```
   Every field has a fixed position -- this is what makes RISC decoding simple.
   The hardware doesn't have to "figure out" how long the instruction is or
   where the fields are. 

---

   ... they just look like alphabet soup until you break down what the 
   processor is actually doing with them. 

   ...

   breakdown of the R-Type instruction fields, reading from right to left (which
   is how the processor often starts looking at it):



---
- `opcode` (7 bits): This is the "Operation Code". It is the main category label
  for the instruction. When the processor reads these 7 bits, it says, "Ah, this
  is a standard Register-toRegister arithmetic operation" or "This is a memory
  load." For all R-Type arithmetic instructions, the opcode is exactly the same.

- `rd` (5 bits): This stands for REGISTER DESTINATION. This tells the processor
  where to save the final result of the calculation. It is 32 bits long because
  RISC-V has 32-general purpose registers, and it takes exactly 5 binary bits to
  count from 0 to 31.

- `funct3` (3 bits): This is a 3-bit "Function" code. Because the `opcode` only
  tells the processor the general category (e.g., "Arithmetic"), the `funct3`
  narrows it down further. For example, it might specify "This is an 
  addition/subtraction type of arithmetic" versus "This is a bitwise shift".

- `rs1` (5 bits): This stands for REGISTER SOURCE 1. It tells the processor
  which register holds the first input value for the math operation.

- `rs2` (5 bits): This stands for REGISTER SOURCE 2. It tells the processor 
  which register holds the second input value.

- `funct7` (7 bits): This is a 7-bit "Function" code, and it acts as the final
  tie-breaker. For example, the `add` and `sub` (subtract) instructions share
  the exact same `opcode` and the exact same `funct3`. The only way the
  processor knows whether to add or subtract is by looking at `funct7`.


WHY SPLIT IT UP LIKE THIS?
   In complex architectures like x86 (Intel/AMD), an instruction can be anywhere
   from 1 to 15 bytes long, and the fields move around. The hardware ahs to do a
   lot of complicated, slow work just to figure out what the instructions say.

   With RISC-V, the fields are in FIXED POSITIONS. When the 32-bit instructions
   hit the processor, the hardware wires can route bits 0-6 directly to the main
   control unit, and bits 7-11 directly to the register file's writee address, 
   all at the exact same time. It makes the physical silicon incredibly fast
   and simple to build. 


```
|  funct7 (7)  |  rs2 (5)  |  rs1  (5)  |  funct3 (3)  |  rd (5)  | opcode (7)  |
```

---


   In the context of this module, is is exactly 32 BITS in each of the 32 bits
   registers. 

   This specific baseline architecture you are studying is called RV32I 
   (Risc-V 32-bit Integer).

   ... breakdown of how that looks hardware-wise:
      * THE NUMBER OF REGISTERS: You have exactly 32 general-purpose registers,
        named x0 through x31.
      * THE SIZE OF EACH REGISTER: Every single one of those registers holds
        exactly 32 BITS (which is 4 bytes) of data.

   In RV32,  a 32-bit chunk of data is officially called a "WORD". This is a 
   super important term to remember because it explains why everything else in
   the datapath lines up so perfectly. Your registers are 32 bits wide, your
   ALU processes 32 bits at a time, and your instructions are exactly 32 bits
   long. Everything is designed around that magic number to keep the hardware
   simple and fast!       



--- ---
   Registers. There are 32 registers. RISC-V names them x0 through x3. We're 
   using the 64-bit version of the RISC-V ISA, so each register holds a 64-bit
   value. 